# Tugas 4 No. 1 — Preprocessing Teks Bahasa Indonesia
**Topik:** Tokenisasi, Stopwords Removal, dan Stemming

**Studi Kasus:** Sistem pencarian dokumen internal Fakultas Teknik UNM. Kumpulan berita dan
pengumuman resmi masih "kotor" (tanda baca, huruf kapital, stopwords, kata berimbuhan) dan perlu
dibersihkan agar siap digunakan untuk sistem temu kembali informasi (Information Retrieval).

**Nama :** _Zahrah Aliyah Zairah_
**NIM  :** _240210501068_


## 0. Instalasi & Import Library

In [10]:
#install sastrawi
!pip install Sastrawi -q

In [11]:
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Inisialisasi Sastrawi (dibuat sekali agar efisien dan dipakai berulang di dalam fungsi)
stopword_factory = StopWordRemoverFactory()
daftar_stopword = set(stopword_factory.get_stop_words())

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

print("Jumlah stopword bawaan Sastrawi:", len(daftar_stopword))
print("Contoh stopword:", list(daftar_stopword)[:10])

Jumlah stopword bawaan Sastrawi: 123
Contoh stopword: ['antara', 'karena', 'adalah', 'kita', 'lagi', 'belum', 'daripada', 'sebetulnya', 'yaitu', 'terhadap']


## 1. Dataset

In [5]:
dokumen = [
    # Dokumen 1
    "Fakultas Teknik UNM mengumumkan bahwa pendaftaran Kuliah Kerja Nyata (KKN) Semester "
    "Genap 2025 akan dibuka mulai tanggal 10 Januari 2025. Mahasiswa diharapkan segera "
    "mendaftarkan diri melalui SIAKAD!",

    # Dokumen 2
    "Workshop \"Kecerdasan Buatan untuk Pemula\" bagi mahasiswa Teknik Informatika akan "
    "diselenggarakan pada tanggal 15-16 Februari 2025 di Laboratorium Komputer Terpadu, "
    "GRATIS untuk 100 peserta pertama.",

    # Dokumen 3
    "Perpustakaan digital kampus kini menyediakan akses ke lebih dari 5.000 jurnal "
    "internasional di bidang teknologi & komputer. Silakan kunjungi portal e-library "
    "kapan saja, 24 jam nonstop!",

    # Dokumen 4
    "Program Studi Teknik Komputer membuka kelas baru mata kuliah Pemrograman Lanjut "
    "dengan kuota 40 mahasiswa pada semester ini. Pendaftaran ditutup 5 hari sebelum "
    "perkuliahan dimulai.",

    # Dokumen 5
    "Jaringan WiFi kampus akan mengalami pemeliharaan rutin pada hari Sabtu, 20 Januari "
    "2025, pukul 08.00-12.00 WITA. Mohon maaf atas ketidaknyamanannya, tim IT sedang "
    "bekerja keras memperbaikinya.",
]

for i, d in enumerate(dokumen, 1):
    print(f"Dokumen {i}: {d}\n")

Dokumen 1: Fakultas Teknik UNM mengumumkan bahwa pendaftaran Kuliah Kerja Nyata (KKN) Semester Genap 2025 akan dibuka mulai tanggal 10 Januari 2025. Mahasiswa diharapkan segera mendaftarkan diri melalui SIAKAD!

Dokumen 2: Workshop "Kecerdasan Buatan untuk Pemula" bagi mahasiswa Teknik Informatika akan diselenggarakan pada tanggal 15-16 Februari 2025 di Laboratorium Komputer Terpadu, GRATIS untuk 100 peserta pertama.

Dokumen 3: Perpustakaan digital kampus kini menyediakan akses ke lebih dari 5.000 jurnal internasional di bidang teknologi & komputer. Silakan kunjungi portal e-library kapan saja, 24 jam nonstop!

Dokumen 4: Program Studi Teknik Komputer membuka kelas baru mata kuliah Pemrograman Lanjut dengan kuota 40 mahasiswa pada semester ini. Pendaftaran ditutup 5 hari sebelum perkuliahan dimulai.

Dokumen 5: Jaringan WiFi kampus akan mengalami pemeliharaan rutin pada hari Sabtu, 20 Januari 2025, pukul 08.00-12.00 WITA. Mohon maaf atas ketidaknyamanannya, tim IT sedang bekerja keras

## 2. Fungsi `preprocess_text(text)`
Urutan tahapan sesuai instruksi tugas:
1. **Case folding** — mengubah seluruh teks menjadi huruf kecil.
2. **Cleaning** — menghapus angka, tanda baca, dan karakter khusus dengan regex.
3. **Tokenisasi** — memecah teks menjadi daftar kata (token).
4. **Stopwords removal** — membuang kata umum yang tidak bermakna penting (Sastrawi).
5. **Stemming** — mengubah kata berimbuhan menjadi kata dasar (Sastrawi).

In [6]:
def preprocess_text(text):
    """
    Pipeline preprocessing teks Bahasa Indonesia:
    case folding -> cleaning -> tokenisasi -> stopwords removal -> stemming

    Parameters
    ----------
    text : str
        Teks mentah (masih kotor).

    Returns
    -------
    dict berisi:
        'tokens_before'  : list token setelah tokenisasi awal (sebelum stopword & stemming)
        'tokens_after'   : list token akhir (setelah stopword removal & stemming)
        'text_after'     : gabungan tokens_after menjadi satu string
    """
    # 1. Case folding
    text = text.lower()

    # 2. Cleaning: hapus angka, tanda baca, dan karakter khusus (hanya sisakan huruf & spasi)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # 3. Tokenisasi
    tokens_before = text.split()

    # 4. Stopwords removal (menggunakan daftar stopword Sastrawi)
    tokens_no_stopword = [t for t in tokens_before if t not in daftar_stopword]

    # 5. Stemming (menggunakan Sastrawi stemmer)
    #    Sastrawi menerima input berupa string, sehingga token digabung dahulu
    text_no_stopword = " ".join(tokens_no_stopword)
    text_stemmed = stemmer.stem(text_no_stopword)
    tokens_after = text_stemmed.split()

    return {
        "tokens_before": tokens_before,
        "tokens_after": tokens_after,
        "text_after": text_stemmed,
    }


# Uji cepat fungsi pada satu contoh kalimat
contoh = preprocess_text(dokumen[0])
print("Token sebelum :", contoh["tokens_before"])
print("Token sesudah :", contoh["tokens_after"])

Token sebelum : ['fakultas', 'teknik', 'unm', 'mengumumkan', 'bahwa', 'pendaftaran', 'kuliah', 'kerja', 'nyata', 'kkn', 'semester', 'genap', 'akan', 'dibuka', 'mulai', 'tanggal', 'januari', 'mahasiswa', 'diharapkan', 'segera', 'mendaftarkan', 'diri', 'melalui', 'siakad']
Token sesudah : ['fakultas', 'teknik', 'unm', 'umum', 'daftar', 'kuliah', 'kerja', 'nyata', 'kkn', 'semester', 'genap', 'buka', 'mulai', 'tanggal', 'januari', 'mahasiswa', 'harap', 'segera', 'daftar', 'diri', 'lalu', 'siakad']


## 3. Penerapan Fungsi pada Ke-5 Dokumen

In [7]:
hasil_preprocessing = [preprocess_text(d) for d in dokumen]

for i, hasil in enumerate(hasil_preprocessing, 1):
    print(f"=== Dokumen {i} ===")
    print("Jumlah token sebelum :", len(hasil["tokens_before"]))
    print("Jumlah token sesudah :", len(hasil["tokens_after"]))
    print()

=== Dokumen 1 ===
Jumlah token sebelum : 24
Jumlah token sesudah : 22

=== Dokumen 2 ===
Jumlah token sebelum : 22
Jumlah token sesudah : 16

=== Dokumen 3 ===
Jumlah token sebelum : 24
Jumlah token sesudah : 20

=== Dokumen 4 ===
Jumlah token sebelum : 23
Jumlah token sesudah : 19

=== Dokumen 5 ===
Jumlah token sebelum : 23
Jumlah token sesudah : 21



## 4. Perbandingan Sebelum & Sesudah Preprocessing

In [8]:
def tampilkan_perbandingan(idx):
    d = dokumen[idx]
    hasil = hasil_preprocessing[idx]
    print(f"########## DOKUMEN {idx+1} ##########")
    print("\n--- TEKS ASLI (SEBELUM) ---")
    print(d)
    print("\n--- TOKEN SEBELUM PREPROCESSING ---")
    print(hasil["tokens_before"])
    print("\n--- TOKEN SESUDAH PREPROCESSING (stopword removal + stemming) ---")
    print(hasil["tokens_after"])
    print("\n--- TEKS SESUDAH PREPROCESSING (digabung kembali) ---")
    print(hasil["text_after"])
    print("\n")

tampilkan_perbandingan(0)  # Dokumen 1
tampilkan_perbandingan(1)  # Dokumen 2

########## DOKUMEN 1 ##########

--- TEKS ASLI (SEBELUM) ---
Fakultas Teknik UNM mengumumkan bahwa pendaftaran Kuliah Kerja Nyata (KKN) Semester Genap 2025 akan dibuka mulai tanggal 10 Januari 2025. Mahasiswa diharapkan segera mendaftarkan diri melalui SIAKAD!

--- TOKEN SEBELUM PREPROCESSING ---
['fakultas', 'teknik', 'unm', 'mengumumkan', 'bahwa', 'pendaftaran', 'kuliah', 'kerja', 'nyata', 'kkn', 'semester', 'genap', 'akan', 'dibuka', 'mulai', 'tanggal', 'januari', 'mahasiswa', 'diharapkan', 'segera', 'mendaftarkan', 'diri', 'melalui', 'siakad']

--- TOKEN SESUDAH PREPROCESSING (stopword removal + stemming) ---
['fakultas', 'teknik', 'unm', 'umum', 'daftar', 'kuliah', 'kerja', 'nyata', 'kkn', 'semester', 'genap', 'buka', 'mulai', 'tanggal', 'januari', 'mahasiswa', 'harap', 'segera', 'daftar', 'diri', 'lalu', 'siakad']

--- TEKS SESUDAH PREPROCESSING (digabung kembali) ---
fakultas teknik unm umum daftar kuliah kerja nyata kkn semester genap buka mulai tanggal januari mahasiswa harap 

## 5.  Jumlah Token Sebelum dan Sesudah Preprocessing beserta Persentase Pengurangan Token

In [9]:
import pandas as pd

rows = []
for i, hasil in enumerate(hasil_preprocessing, 1):
    n_before = len(hasil["tokens_before"])
    n_after = len(hasil["tokens_after"])
    pengurangan = (n_before - n_after) / n_before * 100 if n_before > 0 else 0
    rows.append({
        "Dokumen": f"Dokumen {i}",
        "Token Sebelum": n_before,
        "Token Sesudah": n_after,
        "Pengurangan (%)": round(pengurangan, 2),
    })

df_stat = pd.DataFrame(rows)

total_before = df_stat["Token Sebelum"].sum()
total_after = df_stat["Token Sesudah"].sum()
total_pengurangan = (total_before - total_after) / total_before * 100

df_stat.loc[len(df_stat)] = ["TOTAL", total_before, total_after, round(total_pengurangan, 2)]

df_stat

,Dokumen,Token Sebelum,Token Sesudah,Pengurangan (%)
0,Dokumen 1,24,22,8.33
1,Dokumen 2,22,16,27.27
2,Dokumen 3,24,20,16.67
3,Dokumen 4,23,19,17.39
4,Dokumen 5,23,21,8.70
5,TOTAL,116,98,15.52


## 6. Analisis Singkat

Preprocessing memberikan dampak positif yang signifikan terhadap kualitas data untuk keperluan
sistem information retrieval (IR). Case folding dan cleaning menyeragamkan bentuk penulisan
kata (menghilangkan perbedaan huruf besar/kecil, angka, dan tanda baca yang tidak relevan secara
semantik), sehingga kata yang sama tidak dihitung sebagai term yang berbeda. Stopwords removal
membuang kata-kata fungsi yang sangat sering muncul (seperti "yang", "dan", "akan") namun tidak
memberi informasi pembeda antar dokumen, sehingga bobot term-term penting menjadi lebih menonjol.
Stemming menyatukan variasi bentuk kata berimbuhan (misalnya "mengumumkan" dan "pengumuman")
menjadi satu kata dasar yang sama, sehingga mengurangi redundansi vocabulary dan meningkatkan
recall pada saat pencarian karena query dan dokumen lebih mudah dicocokkan. Secara keseluruhan,
proses ini terbukti mengurangi jumlah token secara berarti sembari
tetap mempertahankan makna inti dokumen, sehingga data menjadi lebih ringkas, konsisten, dan siap
digunakan.